# Session 5: LangGraph - Building Intelligent Agents (55 minutes)

## 🎯 Learning Objectives
- Understand the difference between chains and agents
- Learn LangGraph's state machine approach
- Build agents that use tools
- Create complex decision-making workflows

## 📋 Problem Statement
Our Research Assistant needs to:
- Make decisions dynamically
- Use different tools based on the task
- Handle complex multi-step workflows

## ⏱️ Session Breakdown
- 10 min: Agents vs Chains
- 15 min: LangGraph fundamentals
- 15 min: Building a tool-using agent
- 10 min: Advanced patterns
- 5 min: Recap

---

## 🆓 Using qwen2:0.5b model


## 1. Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize TinyLlama via Ollama (LOCAL, FREE!)
llm = ChatOllama(
    model="qwen2:0.5b",
    temperature=0.7
)

print("✅ Session 5 Setup Complete!")
print("🖥️  Using TinyLlama via Ollama - LOCAL, FREE, NO LIMITS!")

✅ Session 5 Setup Complete!
🖥️  Using TinyLlama via Ollama - LOCAL, FREE, NO LIMITS!


## 2. Chains vs Agents

### Chain (Linear Flow)
```
Input → Step 1 → Step 2 → Step 3 → Output
```
- Predefined sequence
- Always runs same steps
- No decision making

### Agent (Dynamic Flow)
```
                    ┌───────────────┐
                    │               │
Input → Decide ─────┼───► Tool A ───┤
          │         │               │
          ├─────────┼───► Tool B ───┤─── Decide ──► Output
          │         │               │       │
          └─────────┼───► Tool C ───┤       │
                    │               │       │
                    └───────────────┘       │
                            ▲               │
                            └───────────────┘
                             (loop back)
```
- Dynamic decisions
- Uses tools as needed
- Can loop and retry

## 3. LangGraph Concepts

**LangGraph** models agents as **state machines**:

- **State**: The data flowing through the graph
- **Nodes**: Functions that process/modify state
- **Edges**: Connections between nodes (can be conditional)
- **Graph**: The complete workflow

In [2]:
# LangGraph imports
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, Sequence
import operator
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

print("✅ LangGraph imported successfully!")

✅ LangGraph imported successfully!


## 4. Defining State

State is a TypedDict that flows through the graph.

In [3]:
# Simple state definition
class SimpleState(TypedDict):
    """State for a simple agent."""
    input: str           # User's input
    output: str          # Final output
    intermediate: str    # Intermediate results

# More complex state with message history
class AgentState(TypedDict):
    """State for a conversational agent."""
    messages: Annotated[Sequence[BaseMessage], operator.add]
    current_step: str
    tool_calls: list
    final_answer: str

print("📊 State definitions created!")

📊 State definitions created!


## 5. Creating a Simple Graph

In [4]:
# Define node functions
def process_input(state: SimpleState) -> SimpleState:
    """First node: Process the input."""
    print(f"📥 Processing input: {state['input']}")
    return {"intermediate": f"Processed: {state['input']}"}

def generate_response(state: SimpleState) -> SimpleState:
    """Second node: Generate response using LLM."""
    prompt = ChatPromptTemplate.from_template(
        "Respond to this in one sentence: {input}"
    )
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({"input": state["intermediate"]})
    print(f"🤖 Generated response")
    return {"output": response}

# Build the graph
simple_graph = StateGraph(SimpleState)

# Add nodes
simple_graph.add_node("process", process_input)
simple_graph.add_node("generate", generate_response)

# Add edges
simple_graph.set_entry_point("process")
simple_graph.add_edge("process", "generate")
simple_graph.add_edge("generate", END)

# Compile
simple_app = simple_graph.compile()

print("✅ Simple graph compiled!")

✅ Simple graph compiled!


## 6. Running the Graph

In [5]:
# Run the graph
result = simple_app.invoke({"input": "Hello, LangGraph!"})

print("\n📊 Final Result:")
print(f"  Input: {result.get('input')}")
print(f"  Intermediate: {result.get('intermediate')}")
print(f"  Output: {result.get('output')}")

📥 Processing input: Hello, LangGraph!
🤖 Generated response

📊 Final Result:
  Input: Hello, LangGraph!
  Intermediate: Processed: Hello, LangGraph!
  Output: LangGraph is a powerful tool designed for the process of processing data.


## 7. Creating Tools

Tools are functions that agents can call to interact with the world.

In [6]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression.
    
    Args:
        expression: A mathematical expression like '2 + 2' or '10 * 5'
    """
    try:
        # Only allow safe math operations
        allowed = set('0123456789+-*/(). ')
        if all(c in allowed for c in expression):
            result = eval(expression)
            return f"Result: {result}"
        return "Error: Invalid expression"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def search_web(query: str) -> str:
    """Search the web for information.
    
    Args:
        query: The search query
    """
    # Simulated search results
    return f"Search results for '{query}': [Simulated] This is information about {query}. Key facts include..."

@tool
def get_weather(location: str) -> str:
    """Get current weather for a location.
    
    Args:
        location: The city or location name
    """
    # Simulated weather
    return f"Weather in {location}: Sunny, 72°F (22°C), Humidity: 45%"

# List of available tools
tools = [calculator, search_web, get_weather]

print("🛠️ Tools created:")
for t in tools:
    print(f"  - {t.name}: {t.description[:50]}...")

🛠️ Tools created:
  - calculator: Evaluate a mathematical expression.

    Args:
   ...
  - search_web: Search the web for information.

    Args:
       ...
  - get_weather: Get current weather for a location.

    Args:
   ...


## 8. Testing Tools

In [7]:
# Test each tool
print("🧪 Testing tools:\n")

print(f"Calculator: {calculator.invoke('(10 + 5) * 2')}")
print(f"Search: {search_web.invoke('artificial intelligence')}")
print(f"Weather: {get_weather.invoke('San Francisco')}")

🧪 Testing tools:

Calculator: Result: 30
Search: Search results for 'artificial intelligence': [Simulated] This is information about artificial intelligence. Key facts include...
Weather: Weather in San Francisco: Sunny, 72°F (22°C), Humidity: 45%


## 9. Binding Tools to the LLM

In [8]:
# ⚠️ NOTE: Qwen 0.5B doesn't support native tool calling
# We'll implement MANUAL tool routing instead

import re

def get_tool_choice_from_llm(user_query: str) -> tuple:
    """
    Ask the LLM which tool would be best for this query.
    Returns (tool_name, tool_arg)
    """
    prompt = ChatPromptTemplate.from_template("""
    The user is asking: {query}
    
    Available tools:
    1. calculator - for math problems
    2. search_web - for general information
    3. get_weather - for weather queries
    
    Which tool should we use? Respond with ONLY the tool name.""")
    
    chain = prompt | llm | StrOutputParser()
    tool_choice = chain.invoke({"query": user_query}).strip().lower()
    
    # Extract the tool name
    if "calculator" in tool_choice or "math" in user_query.lower():
        return "calculator", user_query
    elif "weather" in tool_choice or "weather" in user_query.lower():
        # Extract location from query
        location = user_query.split("weather")[-1].strip().split("?")[0].strip()
        return "get_weather", location or "San Francisco"
    else:
        return "search_web", user_query

def manual_tool_call(query: str) -> str:
    """Manually select and execute the appropriate tool."""
    tool_name, tool_arg = get_tool_choice_from_llm(query)
    
    print(f"  🛠️ Selected tool: {tool_name}")
    print(f"  📝 Tool input: {tool_arg}")
    
    if tool_name == "calculator":
        result = calculator.invoke(tool_arg)
    elif tool_name == "get_weather":
        result = get_weather.invoke(tool_arg)
    else:
        result = search_web.invoke(tool_arg)
    
    print(f"  ✅ Tool output: {result}")
    return result

# Demo manual tool calling
print("🤖 Manual Tool Calling (for small models):\n")
test_query = "What's 25 * 4?"
print(f"Query: {test_query}")
result = manual_tool_call(test_query)

🤖 Manual Tool Calling (for small models):

Query: What's 25 * 4?
  🛠️ Selected tool: get_weather
  📝 Tool input: What's 25 * 4
  ✅ Tool output: Weather in What's 25 * 4: Sunny, 72°F (22°C), Humidity: 45%


## 10. Building a ReAct Agent

**ReAct** = Reasoning + Acting

The agent:
1. **Reasons** about what to do
2. **Acts** by calling a tool
3. **Observes** the result
4. **Repeats** until done

In [19]:
# Since Qwen 0.5B doesn't support native function calling,
# we'll build a custom ReAct-like agent using manual tool routing

def react_agent_simple(query: str, max_iterations: int = 3):
    """
    Simple ReAct-like agent:
    1. REASON: Decide what to do
    2. ACT: Execute a tool
    3. OBSERVE: Check the result
    4. REPEAT if needed
    """
    print(f"\n🎯 ReAct Agent: {query}")
    print("=" * 60)
    
    messages = [HumanMessage(content=query)]
    
    for iteration in range(max_iterations):
        print(f"\n--- Iteration {iteration + 1} ---")
        
        # REASON: Ask LLM what to do
        reasoning_prompt = ChatPromptTemplate.from_template("""
        Current query: {query}
        
        Do we need a tool for this? (calculator/search/weather/done)
        Respond with just the tool name or 'done'.""")
        
        chain = reasoning_prompt | llm | StrOutputParser()
        decision = chain.invoke({"query": query}).strip().lower()
        
        print(f"🧠 Reasoning: {decision}")
        
        # If done, generate final answer
        if "done" in decision or iteration == max_iterations - 1:
            print(f"\n✅ Final Answer:")
            final_prompt = ChatPromptTemplate.from_template(
                "Answer this briefly: {query}"
            )
            final_chain = final_prompt | llm | StrOutputParser()
            answer = final_chain.invoke({"query": query})
            print(f"  {answer}")
            return answer
        
        # ACT: Execute the appropriate tool
        tool_result = manual_tool_call(query)
        print(f"🔍 Observation: {tool_result}")
        
        # Update for next iteration
        messages.append(AIMessage(content=f"Using tool: {tool_result}"))
    
    return "Completed ReAct loop"

# Test the simple ReAct agent
react_result = react_agent_simple("What is 15 + 27?")

print("\n✅ Simple ReAct Agent created and working!")


🎯 ReAct Agent: What is 15 + 27?

--- Iteration 1 ---
🧠 Reasoning: calculator
  🛠️ Selected tool: get_weather
  📝 Tool input: What is 15 + 27
  ✅ Tool output: Weather in What is 15 + 27: Sunny, 72°F (22°C), Humidity: 45%
🔍 Observation: Weather in What is 15 + 27: Sunny, 72°F (22°C), Humidity: 45%

--- Iteration 2 ---
🧠 Reasoning: done

✅ Final Answer:
  The sum of 15 and 27 equals 42.

✅ Simple ReAct Agent created and working!


## 11. Running the ReAct Agent

In [10]:
# Test the simple ReAct agent with different queries
test_queries = [
    "What is 15 + 27?",
    "What's the weather in New York?",
    "Important facts about artificial intelligence"
]

print("🤖 Simple ReAct Agent Demo (for small models)\n" + "="*60)

for query in test_queries[:1]:  # Run just the first one to save tokens
    react_agent_simple(query)
    print("\n" + "="*60)

🤖 Simple ReAct Agent Demo (for small models)

🎯 ReAct Agent: What is 15 + 27?

--- Iteration 1 ---
🧠 Reasoning: done.

✅ Final Answer:
  \(15 + 27 = 42\).



## 12. Building a Custom Agent with Conditional Edges

In [11]:
from typing import Literal
from langchain_core.messages import ToolMessage

# State for our custom agent
class CustomAgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    next_step: str
    needs_tool: bool  # Track if a tool is needed

def should_continue(state: CustomAgentState) -> Literal["tools", "end"]:
    """Decide whether to continue or end."""
    # Check if the latest message indicates a tool is needed
    return "tools" if state.get("needs_tool", False) else "end"

def call_model(state: CustomAgentState) -> CustomAgentState:
    """Call the LLM and check if it needs a tool."""
    messages = state["messages"]
    last_message = messages[-1].content if messages else ""
    
    # Simple heuristic for small models:
    # If the query has numbers or mentions calculator, we might need a tool
    # If it mentions weather, we might need the weather tool
    # Otherwise it's a general query
    
    needs_tool = False
    
    if any(char.isdigit() for char in last_message) and any(op in last_message for op in ['+', '-', '*', '/', 'add', 'multiply', 'divide']):
        needs_tool = True
    elif "weather" in last_message.lower():
        needs_tool = True
    
    # Get LLM response
    response = llm.invoke(messages)
    
    return {
        "messages": [response],
        "needs_tool": needs_tool
    }

def call_tools(state: CustomAgentState) -> CustomAgentState:
    """Execute tool for the query."""
    messages = state["messages"]
    
    # Get the user's original query
    user_query = ""
    for msg in messages:
        if isinstance(msg, HumanMessage):
            user_query = msg.content
    
    # Use manual tool selection
    if user_query:
        tool_result = manual_tool_call(user_query)
        return {"messages": [ToolMessage(content=tool_result, tool_call_id="manual")]}
    
    return {"messages": []}


## 13. Building the Custom Agent Graph

In [12]:
# Build the custom agent graph
custom_agent = StateGraph(CustomAgentState)

# Add nodes
custom_agent.add_node("agent", call_model)
custom_agent.add_node("tools", call_tools)

# Set entry point
custom_agent.set_entry_point("agent")

# Add conditional edge from agent
custom_agent.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "end": END
    }
)

# Tools always go back to agent
custom_agent.add_edge("tools", "agent")

# Compile
custom_app = custom_agent.compile()

print("✅ Custom agent compiled!")
print("\n📊 Graph structure:")
print("  Entry → Agent → (condition) → Tools → Agent → ...")
print("                → End")

✅ Custom agent compiled!

📊 Graph structure:
  Entry → Agent → (condition) → Tools → Agent → ...
                → End


## 14. Testing the Custom Agent

In [13]:
# Test the custom agent
test_messages = [HumanMessage(content="Calculate 100 / 4 and then tell me the weather in London")]

print("🤖 Custom Agent Execution:\n")

result = custom_app.invoke({"messages": test_messages})

print("📜 Message History:")
for msg in result["messages"]:
    msg_type = type(msg).__name__
    content = msg.content[:100] if msg.content else "[Tool calls]"
    print(f"  [{msg_type}]: {content}...")

🤖 Custom Agent Execution:

  🛠️ Selected tool: calculator
  📝 Tool input: Calculate 100 / 4 and then tell me the weather in London
  ✅ Tool output: Error: Invalid expression
📜 Message History:
  [HumanMessage]: Calculate 100 / 4 and then tell me the weather in London...
  [AIMessage]: The weather in London would be nice. Let's calculate \( \frac{100}{4} \). 

\( 100 \div 4 = 25 \)

S...
  [ToolMessage]: Error: Invalid expression...
  [AIMessage]:  The sky is usually sunny and clear throughout the day....


## 15. Streaming Agent Output

In [14]:
# Stream the agent's execution
print("📡 Streaming Agent Execution:\n")

for event in custom_app.stream(
    {"messages": [HumanMessage(content="What is 50 * 2?")]}
):
    for key, value in event.items():
        print(f"🔸 Node: {key}")
        if "messages" in value:
            for msg in value["messages"]:
                content = msg.content[:80] if msg.content else str(msg.tool_calls if hasattr(msg, 'tool_calls') else "")
                print(f"   Message: {content}")
        print()

📡 Streaming Agent Execution:

🔸 Node: agent
   Message: The answer is 100.

  🛠️ Selected tool: calculator
  📝 Tool input: What is 50 * 2?
  ✅ Tool output: Error: Invalid expression
🔸 Node: tools
   Message: Error: Invalid expression

🔸 Node: agent
   Message: []



## 16. Advanced Pattern: Router Agent

In [15]:
# Router that decides which specialized agent to use
# Since Qwen 0.5B doesn't support structured output,
# we'll parse the LLM response manually

def route_query_manual(query: str) -> dict:
    """Route a query to the appropriate handler using manual parsing."""
    prompt = ChatPromptTemplate.from_template("""
    Analyze this query and decide which category it belongs to:
    - math: Mathematical calculations
    - weather: Weather-related questions  
    - search: Questions requiring web search
    - general: General conversation
    
    Respond with ONLY one of: math, weather, search, or general
    
    Query: {query}
    """)
    
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({"query": query}).strip().lower()
    
    # Parse the route from response
    route = "general"
    reasoning = "General question"
    
    if "math" in response:
        route = "math"
        reasoning = "Mathematical calculation detected"
    elif "weather" in response:
        route = "weather"
        reasoning = "Weather-related question"
    elif "search" in response:
        route = "search"
        reasoning = "Information search required"
    else:
        route = "general"
        reasoning = "General conversation"
    
    return {"route": route, "reasoning": reasoning}

# Test the router
test_queries = [
    "What is 15 * 3?",
    "Is it raining in Tokyo?",
    "Tell me about machine learning",
    "Hello, how are you?"
]

print("🔀 Router Agent Demo\n" + "="*50)
for query in test_queries:
    decision = route_query_manual(query)
    print(f"\nQuery: {query}")
    print(f"  Route: {decision['route']}")
    print(f"  Reason: {decision['reasoning']}")

🔀 Router Agent Demo

Query: What is 15 * 3?
  Route: math
  Reason: Mathematical calculation detected

Query: Is it raining in Tokyo?
  Route: weather
  Reason: Weather-related question

Query: Tell me about machine learning
  Route: math
  Reason: Mathematical calculation detected

Query: Hello, how are you?
  Route: math
  Reason: Mathematical calculation detected


## 17. Checkpointing and Memory

In [16]:
from langgraph.checkpoint.memory import MemorySaver

# Create a checkpointer for persistence
checkpointer = MemorySaver()

# Compile with checkpointer
persistent_agent = custom_agent.compile(checkpointer=checkpointer)

# Thread config for conversation tracking
config = {"configurable": {"thread_id": "user-123"}}

# First message
result1 = persistent_agent.invoke(
    {"messages": [HumanMessage(content="My name is Alice")]},
    config=config
)
print("First interaction:", result1["messages"][-1].content[:100])

# Second message - agent remembers!
result2 = persistent_agent.invoke(
    {"messages": [HumanMessage(content="What's my name?")]},
    config=config
)
print("Second interaction:", result2["messages"][-1].content[:100])

First interaction: Hello! It seems like you've used the phrase "Alice". How can I assist you today?
Second interaction: Your name is Alice.


## 18. Visualizing the Graph (Optional)

In [17]:
# Print graph structure as text (visualization requires graphviz)
print("📊 Agent Graph Structure:\n")
print("""
┌─────────────────────────────────────────────────────┐
│                    ENTRY POINT                       │
└─────────────────────────────────────────────────────┘
                          │
                          ▼
┌─────────────────────────────────────────────────────┐
│                      AGENT                           │
│   (Calls LLM, decides if tools are needed)          │
└─────────────────────────────────────────────────────┘
                          │
              ┌───────────┴───────────┐
              │                       │
       [has tool calls]        [no tool calls]
              │                       │
              ▼                       ▼
┌───────────────────────┐   ┌───────────────────────┐
│        TOOLS          │   │         END           │
│ (Execute tool calls)  │   │  (Return response)   │
└───────────────────────┘   └───────────────────────┘
              │
              └────────────────────────────────────────┐
                                                       │
                          (back to AGENT)              ▲
                                                       │
""")

📊 Agent Graph Structure:


┌─────────────────────────────────────────────────────┐
│                    ENTRY POINT                       │
└─────────────────────────────────────────────────────┘
                          │
                          ▼
┌─────────────────────────────────────────────────────┐
│                      AGENT                           │
│   (Calls LLM, decides if tools are needed)          │
└─────────────────────────────────────────────────────┘
                          │
              ┌───────────┴───────────┐
              │                       │
       [has tool calls]        [no tool calls]
              │                       │
              ▼                       ▼
┌───────────────────────┐   ┌───────────────────────┐
│        TOOLS          │   │         END           │
│ (Execute tool calls)  │   │  (Return response)   │
└───────────────────────┘   └───────────────────────┘
              │
              └────────────────────────────────────────┐


## 📚 Session 5 Recap

### Key Takeaways:

1. **Agents vs Chains:**
   - Chains: Linear, predefined flow
   - Agents: Dynamic, decision-based flow

2. **LangGraph Core Concepts:**
   - **State**: TypedDict flowing through the graph
   - **Nodes**: Functions that process state
   - **Edges**: Connections (conditional or direct)

3. **Tools:**
   - Define with `@tool` decorator
   - Bind to LLM with `.bind_tools()`
   - Execute in tools node

4. **ReAct Pattern:**
   - Reason → Act → Observe → Repeat
   - Use `create_react_agent()` for quick setup

5. **Checkpointing:**
   - Use `MemorySaver()` for persistence
   - Track conversations with `thread_id`

---

### 🔜 Next Session: HuggingFace & Transformers
"We've used cloud APIs. What about local/open-source models?"

In [18]:
print("""
╔═══════════════════════════════════════════════════════════════════════════╗
║                    SESSION 5 COMPLETE! 🎉                                  ║
║                                                                            ║
║  ☕ BREAK TIME - 10 MINUTES ☕                                              ║
║                                                                            ║
║  Next: Session 6 - HuggingFace & Transformers                              ║
║  File: 06_huggingface_transformers.ipynb                                   ║
║                                                                            ║
║  "What if we want to use open-source models?"                              ║
║  "How do we work with HuggingFace models in LangChain?"                    ║
╚═══════════════════════════════════════════════════════════════════════════╝
""")


╔═══════════════════════════════════════════════════════════════════════════╗
║                    SESSION 5 COMPLETE! 🎉                                  ║
║                                                                            ║
║  ☕ BREAK TIME - 10 MINUTES ☕                                              ║
║                                                                            ║
║  Next: Session 6 - HuggingFace & Transformers                              ║
║  File: 06_huggingface_transformers.ipynb                                   ║
║                                                                            ║
║  "What if we want to use open-source models?"                              ║
║  "How do we work with HuggingFace models in LangChain?"                    ║
╚═══════════════════════════════════════════════════════════════════════════╝

